# GVAF — 83개 산업 VARX 추정 (Colab) 🧪

**목표:** 산업별 **실질 부가가치(연도별) 83개 산업** + 외생변수(유가·환율·금리·취업자수·세계교역량)로
**VARX** 모형을 추정해 본다. (먼저 "돌아가는지" 확인 → 이후 Large Bayesian VAR로 확장)

**흐름:** ① 데이터 → ② 정상성 검정·차분 → ③ VARX 추정 → ④ 진단 → ⑤ 예측

> 🟢 **바로 실행:** 기본은 **데모(합성 83산업)** 라 `런타임 → 모두 실행`이면 끝까지 돕니다.
> 🔵 **실데이터:** `CONFIG['DATA_MODE']='csv'` 로 바꾸고 83산업 부가가치 CSV + 외생 CSV를 업로드.

> ⚠️ **미리 알아둘 점:** 83개 산업 × 연간(약 35~40년)이면 추정할 계수가 **표본보다 훨씬 많습니다**
> (83²≈6,900개). 그래서 순수 OLS VARX는 불가능 → 이 노트북은 **수축(ridge)을 자동 적용**해서 돌립니다.
> 이게 바로 **Large (Bayesian) VAR이 필요한 이유**예요. (지금은 "가능성 확인" 단계)

In [ ]:
# Colab 기본 패키지로 충분
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller
from statsmodels.stats.diagnostic import acorr_ljungbox
np.set_printoptions(suppress=True, precision=3)
plt.rcParams['figure.figsize'] = (11, 4); plt.rcParams['axes.grid'] = True
print('setup OK')

## ⚙️ 설정
- `DATA_MODE`: `'demo'`(합성, 즉시 실행) / `'csv'`(실데이터 업로드)
- `N_INDUSTRIES`: 산업 수(데모용). 실데이터면 CSV 열 수가 자동 사용
- `LAGS`: 시차. 연간·소표본이면 **1 권장**
- `RIDGE`: `None`=자동(모수>표본이면 수축 ON) / `0.0`=순수 OLS / `>0`=수동 수축강도

In [ ]:
CONFIG = {
    'DATA_MODE': 'demo',                 # 'demo' | 'csv'
    'N_INDUSTRIES': 83,
    'LAGS': 1,
    'EXOG': ['oil','fx','rate','emp','trade'],   # 유가 환율 금리 취업자수 세계교역량
    'FORECAST_STEPS': 8,                 # 연 단위 예측 기간
    'RIDGE': None,                       # None=자동
    'SEED': 7,
}
CONFIG

## ① 데이터

**실데이터로 돌릴 때 (CSV 2개 업로드)**
- **산업 CSV**: 1열=연도(index), 나머지 83개 열 = 산업별 실질부가가치(예: 십억원). *겹치지 않는 분류.*
- **외생 CSV**: 1열=연도, 열 = `oil, fx, rate, emp, trade`
- 두 파일 모두 **연간**, 같은 연도 범위로 맞추세요.

**어디서 받나 (참고)**
- 83개 산업 부가가치: 한국은행 **국민계정 경제활동별** / **산업연관표 통합중분류(83부문)**.
  - ⚠️ 국민계정 연간 공표는 ~30개 수준, 83개(통합중분류)는 산업연관표(5년+연장) 기반이라
    **연간 시계열은 별도 구성이 필요**합니다. (또는 프로젝트/동기가 받아둔 83산업 데이터를 그대로 사용)
- 외생: 환율·금리·유가→**ECOS**, 취업자수→**경제활동인구조사(KOSIS)**, 세계교역량→**IMF/CPB**.

아래 **데모**는 위 상황(83산업·연간·소표본)을 흉내 낸 합성 데이터라 바로 돕니다.

In [ ]:
def make_demo(n_ind, T=40, seed=7):
    rng = np.random.default_rng(seed)
    idx = pd.period_range('1985', periods=T, freq='Y').to_timestamp()   # 연간
    f = np.cumsum(rng.normal(0.02, 0.02, T))                  # 공통 거시요인
    oil   = 4.0 + np.cumsum(rng.normal(0, 0.10, T))
    fx    = 7.0 + np.cumsum(rng.normal(0, 0.04, T))
    rate  = 2.5 + np.cumsum(rng.normal(0, 0.08, T))
    emp   = 16.0 + np.cumsum(rng.normal(0.01, 0.015, T))
    trade = 5.0 + 0.8*f + np.cumsum(rng.normal(0, 0.06, T))
    X = pd.DataFrame({'oil':oil,'fx':fx,'rate':rate,'emp':emp,'trade':trade}, index=idx)
    load = rng.uniform(0.3, 1.2, n_ind); bx = rng.normal(0, 0.25, (n_ind, 2))
    Y = np.zeros((T, n_ind))
    for i in range(n_ind):
        Y[:, i] = (10 + load[i]*f
                   + bx[i,0]*(oil-oil.mean()) + bx[i,1]*(trade-trade.mean())
                   + np.cumsum(rng.normal(0.02, 0.04, T)))      # 로그 부가가치 수준 (I(1))
    Y = pd.DataFrame(Y, index=idx, columns=[f'ind{i+1:02d}' for i in range(n_ind)])
    return Y, X

def load_csv():
    from google.colab import files
    print('① 산업 부가가치 CSV → ② 외생(oil,fx,rate,emp,trade) CSV 순으로 업로드')
    up = files.upload(); names = list(up.keys())
    Y = pd.read_csv(names[0], index_col=0, parse_dates=True)
    X = pd.read_csv(names[1], index_col=0, parse_dates=True)
    return Y, X

if CONFIG['DATA_MODE'] == 'demo':
    Y_lvl, X_lvl = make_demo(CONFIG['N_INDUSTRIES'], seed=CONFIG['SEED'])
else:
    Y_lvl, X_lvl = load_csv()
print('내생(산업):', Y_lvl.shape, '| 외생:', X_lvl.shape, '| 기간:', Y_lvl.index[0].year, '~', Y_lvl.index[-1].year)
Y_lvl.iloc[:3, :5]

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
Y_lvl.iloc[:, :8].plot(ax=ax[0], legend=False, title='산업 부가가치(로그수준, 일부)')
X_lvl.plot(ax=ax[1], title='외생변수(수준)')
plt.tight_layout(); plt.show()

## ② 정상성 검정 & 차분
VARX는 **정상(stationary)** 시계열을 가정. 수준이 떠도는(단위근) 변수는 **차분**해서 넣어야 함
(안 그러면 가짜 회귀). 산업 부가가치(로그)는 **로그차분=성장률**, 외생은 ADF로 보고 비정상이면 차분.

In [ ]:
def adf_p(s):
    s = pd.Series(s).dropna()
    if len(s) < 8: return np.nan
    try: return adfuller(s, autolag='AIC')[1]
    except Exception: return np.nan

# 외생 정상성 요약
rows=[]
for c in X_lvl.columns:
    p0,p1 = adf_p(X_lvl[c]), adf_p(X_lvl[c].diff())
    order='I(0)' if (p0==p0 and p0<0.05) else ('I(1)' if (p1==p1 and p1<0.05) else 'I(2)+?')
    rows.append((c, round(p0,3) if p0==p0 else None, round(p1,3) if p1==p1 else None, order))
display(pd.DataFrame(rows, columns=['외생','ADF p(수준)','ADF p(차분)','적분차수']))

# 산업: 수준에서 정상인 비율(참고) → 일괄 로그차분
ind_p0 = np.array([adf_p(Y_lvl[c]) for c in Y_lvl.columns])
print('산업 부가가치: 수준에서 정상(I(0)) 비율 = %.0f%% → 전부 1차차분(성장률)으로' % (100*np.mean(ind_p0<0.05)))

dY = Y_lvl.diff().dropna()
Xs = pd.DataFrame(index=X_lvl.index)
for c in X_lvl.columns:
    p0 = adf_p(X_lvl[c]); Xs[c] = X_lvl[c] if (p0==p0 and p0<0.05) else X_lvl[c].diff()
Xs = Xs.dropna()
common = dY.index.intersection(Xs.index); dY, Xs = dY.loc[common], Xs.loc[common]
print('정상화 후 — 내생(성장률):', dY.shape, '| 외생:', Xs.shape)

## ③ VARX 추정
각 산업 성장률 ~ [상수 + 모든 산업의 과거 성장률(시차 p) + 동시기 외생변수].

**핵심:** 방정식당 계수 ≈ `1 + N·p + k`. N=83·p=1·k=5면 **89개** — 그런데 연간 표본은 ~40개뿐.
**모수 ≫ 표본** 이라 순수 OLS는 불가 → **수축(ridge) 자동 적용**(스케일 보정, 상수항은 제외).

In [ ]:
def fit_varx(Yarr, Xarr, p=1, ridge=None):
    T, n = Yarr.shape; k = Xarr.shape[1]
    Z, tgt = [], []
    for t in range(p, T):
        row=[1.0]
        for l in range(1, p+1): row += list(Yarr[t-l])
        row += list(Xarr[t])
        Z.append(row); tgt.append(Yarr[t])
    Z, Yt = np.array(Z), np.array(tgt)
    m, neff = Z.shape[1], Z.shape[0]
    ZtZ = Z.T @ Z
    over = (m >= neff)
    base = (0.1 if over else 0.0) if ridge is None else float(ridge)   # 과대모수면 자동 수축
    scale = np.trace(ZtZ)/m
    lam = base*scale
    P = np.eye(m); P[0,0] = 0.0                # 상수항은 수축 제외
    B = np.linalg.solve(ZtZ + lam*P, Z.T@Yt)
    resid = Yt - Z@B
    return dict(B=B, Z=Z, Yt=Yt, resid=resid, p=p, n=n, k=k, m=m, neff=neff, lam=lam, over=over, base=base)

p = CONFIG['LAGS']
fit = fit_varx(dY.values, Xs.values, p=p, ridge=CONFIG['RIDGE'])
print('방정식당 계수 %d개  vs  표본 %d개  (내생 %d · 외생 %d · 시차 %d)' % (fit['m'], fit['neff'], fit['n'], fit['k'], p))
if fit['over']:
    print('⚠️ 모수 ≥ 표본 → 순수 OLS 불가. 수축 자동 ON (base=%.2f, λ=%.4f)' % (fit['base'], fit['lam']))
else:
    print('표본 충분 → OLS')
ss_res=(fit['resid']**2).sum(0); ss_tot=((fit['Yt']-fit['Yt'].mean(0))**2).sum(0)
r2 = 1 - ss_res/np.where(ss_tot==0,np.nan,ss_tot)
print('산업별 R²(참고) 중앙값 %.2f  ※ 과대모수라 in-sample 적합은 과신 금물' % np.nanmedian(r2))

## ④ 진단
- **안정성**: 동반행렬 최대 |고유값| < 1 → 예측 발산 안 함
- **잔차 자기상관**: Ljung-Box p>0.05 → 백색잡음(설명 잘 됨)

In [ ]:
n, p, B = fit['n'], fit['p'], fit['B']
A_list, off = [], 1
for l in range(p): A_list.append(B[off:off+n,:].T); off += n
comp = A_list[0] if p==1 else np.vstack([np.hstack(A_list),
        np.hstack([np.eye(n*(p-1)), np.zeros((n*(p-1), n))])])
eig = np.abs(np.linalg.eigvals(comp))
print('안정성: 최대 |고유값| = %.3f → %s' % (eig.max(), '안정 ✅' if eig.max()<1 else '불안정 ⚠️ (수축 강화 필요)'))
print('잔차 백색잡음(대표 산업):')
for j in range(min(3,n)):
    try:
        pv = acorr_ljungbox(fit['resid'][:,j], lags=[4])['lb_pvalue'].iloc[0]
        print('  산업%d  Ljung-Box(4) p=%.3f  %s' % (j+1, pv, 'OK' if pv>0.05 else '자기상관 잔존'))
    except Exception as e: print('  산업%d skip (%s)'%(j+1,e))

## ⑤ 예측 (가능성 확인)
외생변수 미래 경로를 주면 산업별 성장률 → 부가가치 수준을 예측. (데모는 최근 평균 변화 유지)

In [ ]:
h = CONFIG['FORECAST_STEPS']
X_future = np.tile(Xs.values[-min(10,len(Xs)):].mean(0), (h,1))
def forecast(fit, Yd, Xf):
    p,B = fit['p'], fit['B']; hist=list(Yd[-p:]); out=[]
    for t in range(len(Xf)):
        row=[1.0]
        for l in range(1,p+1): row += list(hist[-l])
        row += list(Xf[t]); yh=np.array(row)@B; out.append(yh); hist.append(yh)
    return np.array(out)
fc = forecast(fit, dY.values, X_future)
fc_lvl = Y_lvl.iloc[-1].values + np.cumsum(fc, axis=0)
xh=np.arange(len(Y_lvl)); xf=np.arange(len(Y_lvl), len(Y_lvl)+h)
fig, ax = plt.subplots(1,2, figsize=(12,4))
for j in range(min(5,n)):
    ln, = ax[0].plot(xh, Y_lvl.iloc[:,j].values, lw=1); ax[0].plot(xf, fc_lvl[:,j],'--',color=ln.get_color())
ax[0].set_title('산업 부가가치(로그수준)+예측(점선)')
ax[1].bar(range(n), fc.mean(0)); ax[1].set_title('산업별 평균 예측 성장률(83개)')
plt.tight_layout(); plt.show()
print('✅ 83개 산업 VARX 파이프라인이 끝까지 돌았습니다 — 가능성 확인')

## ✅ 결론 & 다음 단계

**확인된 것:** 83개 산업 VARX가 **끝까지 돈다**(데이터 로드→차분→추정→진단→예측).

**솔직한 한계 (중요):** 83산업 × 연간(~40)이면 **계수(89)가 표본(~39)보다 많다.**
- 이 노트북은 **ridge 수축**으로 억지로 돌린 것 — in-sample 적합·개별 계수는 **신뢰하면 안 됨**.
- 제대로 하려면 → **Large Bayesian VAR**(Minnesota + 산업연관표 prior). 같은 골격에 ① 똑똑한 기본값으로 수축, ② 사후분포 → **신뢰구간 팬차트** 를 얹으면 됨.
- 또는 **분기 자료**(시점↑)로 표본을 늘리면 더 안정적.

**실데이터로 돌리려면:** `CONFIG['DATA_MODE']='csv'` → 83산업 부가가치 CSV + 외생 CSV 업로드 → 모두 실행.

**경로:** (지금) VARX+ridge 가능성 확인 → **Large Bayesian VARX**(팬차트) → 제안서 Block 2.